# 30 孟加拉国 DIA 冻结模型推理（V2）

运行 Notebook 19 冻结的完整八候选模型体系，并逐端点采用最终注册模型：P=DreaMS，B/NOEC=Transformer B，EC50/LC50=binned MS/MS。应用冻结校准和拒绝阈值，分别导出 Pos/Neg 结果。

建议使用 `env_dreams` 内核和 CUDA。整个过程无需手工指定检查点。

In [ ]:
from pathlib import Path
import sys,json
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'pbt_msms').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import torch
print('ROOT =',ROOT); print('Python =',sys.executable); print('device =','cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
from pbt_msms.bangladesh_dia_external import paths
p=paths(ROOT)
prepare=json.loads(p['prepare_manifest'].read_text(encoding='utf-8'))
embed=json.loads(p['embed_manifest'].read_text(encoding='utf-8'))
assert prepare['status']=='READY_FOR_DREAMS_EMBEDDING'
assert embed['status']=='READY_FOR_FROZEN_MODEL_INFERENCE'
print('Notebook 29 gate: PASS')


In [ ]:
# 将依次运行 binned、DreaMS heads、Set Transformer 和 Transformer A-E；可能需要较长时间。
from pbt_msms.bangladesh_dia_external import run_prediction
manifest=run_prediction(ROOT)
print(json.dumps(manifest,ensure_ascii=False,indent=2))
assert manifest['status']=='PREDICTIONS_FROZEN'


In [ ]:
import pandas as pd,numpy as np
final=pd.read_csv(p['prediction']/'30_final_selected_model_predictions.csv.gz')
display(final.groupby(['mode','task']).agg(n=('spectrum_id','size'),retained=('retained','sum'),mean_prediction=('y_prediction','mean'),median_uncertainty=('uncertainty_score','median')).reset_index())
assert final.groupby('spectrum_id')['task'].nunique().eq(5).all()
assert np.isfinite(final['y_score_calibrated']).all()
print('PASS：第30步预测已冻结。输出目录：',p['prediction'])
